## 1. Import required Libraries

In [2]:
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.metrics import roc_auc_score, f1_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

SEED = 983
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

## 2. Import data

In [3]:
curr_path = Path.cwd()
Output_Datasets_path = curr_path.parent.parent / 'Output_Datasets'

In [4]:
X_train_clean_path = Output_Datasets_path / 'X_train_clean.npy'
Y_train_clean_path = Output_Datasets_path / 'Y_train_clean.npy'
X_val_clean_path = Output_Datasets_path / 'X_val_clean.npy'
Y_val_clean_path = Output_Datasets_path / 'Y_val_clean.npy'

In [7]:
print("Loading X_train_clean...")
X_train_clean = np.load(X_train_clean_path, allow_pickle=True)
print("Loading Y_train_clean...")
Y_train_clean = np.load(Y_train_clean_path, allow_pickle=True)
print("Successfully loaded X_train_clean and Y_train_clean")

Loading X_train_clean...
Loading Y_train_clean...
Successfully loaded X_train_clean and Y_train_clean


In [9]:
# print("Loading X_val_clean and Y_val_clean...")
# X_val_clean = np.load(X_val_clean_path, allow_pickle=True)
# Y_val_clean = np.load(Y_val_clean_path, allow_pickle=True)
# print("Successfully loaded X_val_clean and Y_val_clean")

In [147]:
print(f'Shape of X_train_clean: {X_train_clean.shape}')
#print(f'Shape of X_val_clean: {X_val_clean.shape}')

Shape of X_train_clean: (17090, 1000, 12)


In [148]:
print(f'Shape of Y_train_clean: {Y_train_clean.shape}')
#print(f'Shape of Y_val_clean: {Y_val_clean.shape}')

Shape of Y_train_clean: (17090, 5)


In [152]:
X_train_clean = X_train_clean[:1000, :, :]
Y_train_clean = Y_train_clean[:1000, :]
print(f'Shape of X_train_clean: {X_train_clean.shape}')
print(f'Shape of Y_train_clean: {Y_train_clean.shape}')

Shape of X_train_clean: (1000, 1000, 12)
Shape of Y_train_clean: (1000, 5)


In [153]:
print(Y_train_clean[0])
print(Y_train_clean[290])

[0 0 0 1 0]
[1 0 0 0 0]


In [154]:
X_train_clean[0].shape

(1000, 12)

In [155]:
torch.tensor(X_train_clean[0,:,0]).shape

torch.Size([1000])

## 3. PyTorch DataSet 

In [156]:
class EcgDataset(Dataset):
    def __init__(self, inputs, target, column=0):
        self.inputs = inputs
        self.target = target
        self.column = column

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return torch.tensor(self.inputs[idx, :, column], dtype=torch.float32), torch.tensor(self.target[idx], dtype=torch.float32) # imp mistake: I tried to give 2d index for a 3d vector

In [157]:
print('Preparing dataset...')
train_dataset = EcgDataset(inputs=X_train_clean, target=Y_train_clean)
# val_dataset = EcgDataset(X_val_clean, Y_val_clean)
print('Completed dataset preparation!')

Preparing dataset...
Completed dataset preparation!


## 4. PyTorch Dataloader

In [158]:
BATCH_SIZE = 30

In [159]:
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
# val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=True)

In [160]:
for i, x in enumerate(train_dataloader):
    if i == 2:
        break
    print(x[0])
    print(x[1])

tensor([[-0.4866, -0.4866, -0.5052,  ..., -0.1509, -0.1944, -0.3374],
        [-0.0203,  0.0108,  0.0791,  ...,  0.0854,  0.0045, -0.0017],
        [-0.1447, -0.1385, -0.1011,  ..., -0.2193, -0.1882, -0.3312],
        ...,
        [-0.9156, -0.8907, -0.1633,  ..., -0.2006, -0.1820, -0.2441],
        [-0.2317, -0.1758, -0.1011,  ..., -0.6171, -0.5861, -0.4493],
        [ 0.1662,  0.2035,  0.2283,  ..., -0.9031, -0.8285, -0.6047]])
tensor([[1., 0., 0., 1., 0.],
        [0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 1.],
        [1., 0., 0., 0., 0.],
        [0., 0., 0., 1., 0.],
        [1., 0., 0., 1., 0.],
        [0., 0., 0., 1., 0.],
        [0., 0., 0., 1., 0.],
        [0., 1., 0., 0., 1.],
        [0., 0., 0., 1., 0.],
        [1., 0., 0., 0., 0.],
        [0., 0., 0., 1., 0.],
        [0., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0.],
        [1., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0.],
        [0., 1., 1., 0., 0.],
        [0., 0., 0., 1., 0.],
        [0., 0., 1., 0., 0

## 4. Bidirectional LSTM Model

In [161]:
SIGNAL_SIZE = 1 # 1 feature per time step (single lead). Change to 12 if passing all 12 leads.
NUM_LAYERS = 2
OUTPUT_DIM = 5
HIDDEN_DIM = 90
EMBED_DIM = 126
DROPOUT = 0.2

In [162]:
class BiLSTM(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=HIDDEN_DIM, embed_dim=EMBED_DIM, output_dim=OUTPUT_DIM, num_layers=NUM_LAYERS, dropout=0.25):
        super(BiLSTM, self).__init__()
        
        # Projects each time step's feature(s) into embed_dim
        self.proj = nn.Linear(input_dim, embed_dim)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bias=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, signal):
        # Ensure 3D shape: [batch_size, seq_len, input_dim]
        if signal.ndim == 2:
            signal = signal.unsqueeze(-1)  # [batch_size, 1000] -> [batch_size, 1000, 1]
            
        # [batch_size, seq_len, 1] -> [batch_size, seq_len, embed_dim]
        projected = self.dropout(torch.relu(self.proj(signal)))
        
        # lstm_out: [batch_size, seq_len, hidden_dim*2]
        # hidden:   [num_layers*2, batch_size, hidden_dim]
        lstm_out, (hidden, cell) = self.lstm(projected)
        
        # Concatenate forward and backward final hidden state
        hidden_last = torch.cat((hidden[-2, :, :], hidden[-1, :, :]), dim=1)  # [batch_size, hidden_dim*2]
        logits = self.fc(self.dropout(hidden_last))
        
        return logits


In [163]:
lstmModel = BiLSTM(input_dim=SIGNAL_SIZE,
                   embed_dim=EMBED_DIM,
                   hidden_dim=HIDDEN_DIM,
                   output_dim=OUTPUT_DIM,
                   num_layers=NUM_LAYERS,
                   dropout=DROPOUT)
print(lstmModel)

BiLSTM(
  (proj): Linear(in_features=1, out_features=126, bias=True)
  (lstm): LSTM(126, 90, num_layers=2, batch_first=True, dropout=0.2, bidirectional=True)
  (fc): Linear(in_features=180, out_features=5, bias=True)
  (dropout): Dropout(p=0.2, inplace=False)
)


## 5. Training

### 5.1 Bi-LSTM Training

In [164]:
LEARNING_RATE = 1e-3
EPOTCHS = 5

In [165]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(lstmModel.parameters(), lr = LEARNING_RATE)

In [177]:
def train_lstm_model(model, datasetloader, criterion, optimizer, device):
    model.to(device)
    model.train()
    train_loss, total_samples, total_elememts = 0, 0, 0
    correct_exact, correct_elements = 0, 0
    correct_element1, correct_element2, correct_element3, correct_element4, correct_element5 = 0,0,0,0,0
    
    
    for x_batch, y_batch in datasetloader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        logits = model(x_batch)
        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()


        probs = torch.sigmoid(logits)
        preds = (probs > 0.6).float()

        exact_matches = (preds == y_batch).all(dim=1)

        correct_exact += exact_matches.sum().item()
        correct_elements += (preds == y_batch).sum().item()
        correct_element1 += (preds[:, 0] == y_batch[:, 0]).sum().item()
        correct_element2 += (preds[:, 1] == y_batch[:, 1]).sum().item()
        correct_element3 += (preds[:, 2] == y_batch[:, 2]).sum().item()
        correct_element4 += (preds[:, 3] == y_batch[:, 3]).sum().item()
        correct_element5 += (preds[:, 4] == y_batch[:, 4]).sum().item()

        total_samples += y_batch.shape[0]
        total_elememts += y_batch.numel()
        train_loss += loss.item() * y_batch.shape[0]
        
        #print(total_samples)

    avg_train_loss = train_loss / total_samples
    exect_acc = correct_exact / total_samples
    elements_acc = correct_elements / total_elememts
    

    return avg_train_loss, exect_acc, elements_acc, [correct_element1, correct_element2, correct_element3, correct_element4, correct_element5]

In [ ]:
print('Training model....')
for epoch in range(1, EPOTCHS+1):

    train_loss, exect_train_accuracy, element_wise_combined_train_acc, element_wise_each_train_acc = train_lstm_model(lstmModel, train_dataloader, criterion, optimizer, device)

    # test_loss, test_accuracy = evaluate_model(lstmModel, test_loader, criterion, device)

    # print(f'Epoch [{epoch}/{EPOTCHS}] | Train Loss: {train_loss:.3f} | Train Acc: {train_accuracy*100:.3f}% | Test Loss: {test_loss:.3f} | Test Acc: {test_accuracy*100:.3f}%')

    print(f'Epoch [{epoch}/{EPOTCHS}] | Train Loss: {train_loss:.3f} | Exect Train Acc: {exect_train_accuracy*100:.3f}% | Element Wise Train Loss: {element_wise_combined_train_acc*100:.3f}%')
    print(f'\tElement wise loss:\t\nTarget1 Acc: {element_wise_each_train_acc[0]}\t\nTarget2 Acc: {element_wise_each_train_acc[1]}\t\nTarget3 Acc: {element_wise_each_train_acc[2]}\t\nTarget4 Acc: {element_wise_each_train_acc[3]}\t\nTarget5 Acc: {element_wise_each_train_acc[4]}\n')

Training model....
Epoch [1/5] | Train Loss: 0.494 | Exect Train Acc: 24.200% | Element Wise Train Loss: 76.540%
Element wise loss:
Target1 Acc: 795
Target2 Acc: 892
Target3 Acc: 827
Target4 Acc: 535
Target5 Acc: 778

Epoch [2/5] | Train Loss: 0.495 | Exect Train Acc: 20.400% | Element Wise Train Loss: 75.980%
Element wise loss:
Target1 Acc: 795
Target2 Acc: 892
Target3 Acc: 827
Target4 Acc: 506
Target5 Acc: 779

Epoch [3/5] | Train Loss: 0.491 | Exect Train Acc: 22.900% | Element Wise Train Loss: 76.140%
Element wise loss:
Target1 Acc: 795
Target2 Acc: 892
Target3 Acc: 827
Target4 Acc: 514
Target5 Acc: 779

Epoch [4/5] | Train Loss: 0.489 | Exect Train Acc: 34.100% | Element Wise Train Loss: 77.660%
Element wise loss:
Target1 Acc: 796
Target2 Acc: 892
Target3 Acc: 827
Target4 Acc: 587
Target5 Acc: 781

Epoch [5/5] | Train Loss: 0.502 | Exect Train Acc: 22.700% | Element Wise Train Loss: 76.400%
Element wise loss:
Target1 Acc: 796
Target2 Acc: 892
Target3 Acc: 828
Target4 Acc: 526
Targ

```Model score
Training model over 1000 rows....
Epoch [1/5] | Train Loss: 0.494 | Exect Train Acc: 24.200% | Element Wise Train Loss: 76.540%
Element wise loss:
Target1 Acc: 795
Target2 Acc: 892
Target3 Acc: 827
Target4 Acc: 535
Target5 Acc: 778

Epoch [2/5] | Train Loss: 0.495 | Exect Train Acc: 20.400% | Element Wise Train Loss: 75.980%
Element wise loss:
Target1 Acc: 795
Target2 Acc: 892
Target3 Acc: 827
Target4 Acc: 506
Target5 Acc: 779

Epoch [3/5] | Train Loss: 0.491 | Exect Train Acc: 22.900% | Element Wise Train Loss: 76.140%
Element wise loss:
Target1 Acc: 795
Target2 Acc: 892
Target3 Acc: 827
Target4 Acc: 514
Target5 Acc: 779

Epoch [4/5] | Train Loss: 0.489 | Exect Train Acc: 34.100% | Element Wise Train Loss: 77.660%
Element wise loss:
Target1 Acc: 796
Target2 Acc: 892
Target3 Acc: 827
Target4 Acc: 587
Target5 Acc: 781

Epoch [5/5] | Train Loss: 0.502 | Exect Train Acc: 22.700% | Element Wise Train Loss: 76.400%
Element wise loss:
Target1 Acc: 796
Target2 Acc: 892
Target3 Acc: 828
Target4 Acc: 526
Target5 Acc: 778

```

## 6. Evaluation

## 7. Visualization